# Acrobot C — exact fast simulation and resumable training (v3)

Run this notebook in a fresh CPU runtime with the same Google account.
Before starting, terminate the old v2 runtime via Runtime → Disconnect and delete runtime.
This prevents any orphan v2 training process competing for CPU.

Only C runs: 3 seeds × 600 episodes, then clean and matched-noise evaluation
(100 episodes each). No reduction in layers, batch size, updates or noise.
A/B remain in `SC4055_Acrobot_Recovery/experiment_v2` and are not modified.
C uses the separate `SC4055_Acrobot_Recovery/experiment_C_v3` directory.

Every completed training episode saves optimizer, target model, replay buffer and
random states. An interruption repeats at most the current unsaved episode.
Each seed prints saved episodes and a training heartbeat about every 30 seconds
between completed updates. First-use compilation may delay the heartbeat.
Reopen this same notebook and Run all to resume. Do not run two copies at once.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, subprocess, shutil, base64, hashlib, zipfile, io, signal
import shlex
for row in subprocess.check_output(['ps', '-eo', 'pid,args'], text=True).splitlines()[1:]:
    parts = shlex.split(row)
    if '-m' in parts:
        pos = parts.index('-m')
        if parts[pos+1:pos+3] == ['sc4055_project.acrobot', 'train']:
            raise RuntimeError('A trainer is still running. Disconnect and delete the old runtime before Run all.')
drive.mount('/content/drive')
PERSIST = Path('/content/drive/MyDrive/SC4055_Acrobot_Recovery')
PERSIST.mkdir(parents=True, exist_ok=True)
SOURCE_B64 = '...'
EXPECTED_SHA256 = 'aaee6cf59189769bdfc087b0b1a6b0dc2af0b9dd6f9d4f447d85957c3b206627'

source = base64.b64decode(SOURCE_B64)
assert hashlib.sha256(source).hexdigest() == EXPECTED_SHA256
source_file = PERSIST/'source_C_v3.zip'
if source_file.exists() and hashlib.sha256(source_file.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError('A different v3 source exists; do not mix source versions')
source_file.write_bytes(source)
assert hashlib.sha256(source_file.read_bytes()).hexdigest() == EXPECTED_SHA256
ROOT = Path('/content/acrobot_c_v3/SC4055-Project')
with zipfile.ZipFile(io.BytesIO(source)) as archive:
    for member in archive.infolist():
        target = (ROOT.parent/member.filename).resolve()
        if target != ROOT and ROOT not in target.parents:
            raise ValueError('Unsafe archive path')
    archive.extractall(ROOT.parent)
print('C v3 source verified on Drive. A/B were not changed.')


In [ ]:
def visible_run(command, **kwargs):
    # Linux/Colab: a new session owns the entire runner + trainer process tree.
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True, **kwargs)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, command)
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            pass
        # The runner may exit before its training child; kill any remainder.
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait()
        raise
    finally:
        process.stdout.close()

visible_run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.13.7'])
UV = shutil.which('uv')
assert UV, 'uv not on PATH'
ENV = os.environ.copy()
ENV.update(MPLBACKEND='Agg', JAX_PLATFORMS='cpu', PYTHONUNBUFFERED='1',
           UV_PYTHON_INSTALL_DIR='/content/acrobot_python', UV_CACHE_DIR='/content/acrobot_uv_cache')
visible_run([UV, 'python', 'install', '3.12', '--no-bin'], env=ENV)
if not (ROOT / '.venv/bin/python').exists():
    visible_run([UV, 'venv', '--python', '3.12', str(ROOT / '.venv')], env=ENV)
PY = str(ROOT / '.venv/bin/python')
ENV['PATH'] = str(ROOT / '.venv/bin') + os.pathsep + ENV['PATH']
visible_run([UV, 'pip', 'install', '--python', PY, 'swig>=4.2'], env=ENV)
visible_run([UV, 'pip', 'install', '--python', PY, '-e', '.', '--group', 'test'], cwd=ROOT, env=ENV)

def run(*args):
    visible_run([PY, '-u', *map(str, args)], cwd=ROOT, env=ENV)

run('-m', 'pytest', 'tests/sc4055_project/tasks/test_acrobot.py', 'tests/sc4055_project/tasks/test_acrobot_recovery_v3.py', '-q')
environment = subprocess.check_output([UV, 'pip', 'freeze', '--python', PY], env=ENV, text=True)
(PERSIST / 'environment_C_v3.txt').write_text(environment)
print('环境和测试已通过。')


## Run C only
The source/protocol lock refuses mismatched versions. Completed seeds are skipped.
The old incomplete C seed has no recoverable training state and is not imported.


In [ ]:
OUT = PERSIST/'experiment_C_v3'
run('analysis/acrobot/recoverable.py', '--out', OUT, '--stage', 'C')
print('C complete; saved to', OUT)
print('A/B remain in', PERSIST/'experiment_v2')


## Optional export
All files already persist on Drive. Run after completion or stopping.

In [ ]:
OUT = PERSIST/'experiment_C_v3'
if OUT.exists():
    archive = shutil.make_archive(str(PERSIST/'Acrobot_C_v3_results'), 'zip',
                                 root_dir=PERSIST, base_dir=OUT.name)
    print('Saved:', archive)
    from google.colab import files
    files.download(archive)
